# CPSA 07 — build the hand-crop dataset and train the stage-2 hand model

Stage 2 of the two-stage design: a hand landmark model that only sees a square
crop around one hand.  Stage 1 (the half-body / full-frame body model) finds the
person and the hands; this notebook builds the data and the model for the crops.

Sources (both downloaded here, no Kaggle dataset attached):

* **HaGRIDv2** — official `annotations_with_landmarks` JSON (21 MediaPipe
  landmarks per hand) plus the 30k-image 384p sample
  (`cj-mills/hagrid-sample-30k-384p`).  1,769 peace frames and the other 16
  gestures are in the sample; every annotated hand becomes one crop.
  License: CC BY-SA 4.0 variant (share-alike, attribution).
* **Ultralytics hand-keypoints** — 26,768 images with 21-keypoint YOLO labels,
  re-cropped with the same geometry.
  License: **CC BY-NC-SA 4.0 — non-commercial.**  Set `USE_ULTRALYTICS = False`
  to build a HaGRID-only dataset that is not encumbered by the NC clause, and
  do not publish a dataset that mixes the two without clearing that license.

The crop geometry lives in `cpsa_ml.datasets.hands`: the hand box is enlarged
1.6-2.2x, the centre and the size are jittered (seeded per sample, so the build
is reproducible and resumable), the window is kept square, and the result is
resized to 192 px.  One class `hand`, 21 keypoints, identity `flip_idx` (a
mirrored crop keeps every landmark on the same finger joint); left and right
hands share the class and the handedness is only counted in `stats.json`.

Training reuses `cpsa_ml.train` with the DPU-friendly LeakyReLU(26/256) config
`configs/yolov8n-hand-lrelu.yaml`, `yolov8n`, `imgsz=192`, batch 64 on 2x T4.

Set `COMMIT_SHA` below to the exact repository commit for this run.  Outputs:
`/kaggle/working/hand21` (dataset, re-publish as `fabiocabeccia/cpsa-hand21`)
and `/kaggle/working/hand-model` (weights + metadata, re-publish as
`fabiocabeccia/cpsa-hand-model`).


In [ ]:
import subprocess
import sys

# Pin the exact CPSA_2026 commit for a reproducible Kaggle run.
COMMIT_SHA = "22ae227b6074c7e54288197dc9de4c5160120445"

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"cpsa-ml @ git+https://github.com/fcabecciaw/CPSA_2026@{COMMIT_SHA}#subdirectory=ml",
    ],
    check=True,
)

In [ ]:
import subprocess
from pathlib import Path

DATA = Path("/tmp/hand_data")  # outside /kaggle/working: not part of the output
DATA.mkdir(parents=True, exist_ok=True)

URLS = {
    "hagrid-384p.zip": (
        "https://huggingface.co/datasets/cj-mills/hagrid-sample-30k-384p"
        "/resolve/main/hagrid-sample-30k-384p.zip"
    ),
    "annotations.zip": (
        "https://rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru/datasets"
        "/hagrid_v2/annotations_with_landmarks/annotations.zip"
    ),
    "hand-keypoints.zip": (
        "https://github.com/ultralytics/assets/releases/download/v0.0.0/hand-keypoints.zip"
    ),
}

# ~2.1 GB in total; curl -C - resumes a download if the session dies.
for name, url in URLS.items():
    archive = DATA / name
    if not archive.is_file():
        subprocess.run(
            ["curl", "-sSL", "--retry", "3", "-C", "-", "-o", str(archive), url], check=True
        )
    print(name, round(archive.stat().st_size / 1e6, 1), "MB")

In [ ]:
import zipfile
from pathlib import Path

# The 384p sample is one top-level folder with one sub-directory per gesture.
with zipfile.ZipFile(DATA / "hagrid-384p.zip") as archive:
    archive.extractall(DATA / "hagrid")
HAGRID_ROOT = DATA / "hagrid" / "hagrid-sample-30k-384p" / "hagrid_30k"
GESTURES = sorted(p.name.split("train_val_")[-1] for p in HAGRID_ROOT.iterdir() if p.is_dir())

# Only the annotation files of the gestures present in the sample: the whole
# tree is ~1.5 GB and most of it is never read.
annotations_zip = zipfile.ZipFile(DATA / "annotations.zip")
members = [
    name
    for name in annotations_zip.namelist()
    if name.startswith("annotations/") and name.endswith(".json") and Path(name).stem in GESTURES
]
annotations_zip.extractall(DATA / "hagrid_annotations", members=members)
ANNOTATIONS = DATA / "hagrid_annotations" / "annotations"

ULTRALYTICS_ROOT = DATA / "hand-keypoints"
if not (ULTRALYTICS_ROOT / "labels" / "train").is_dir():
    with zipfile.ZipFile(DATA / "hand-keypoints.zip") as archive:
        archive.extractall(ULTRALYTICS_ROOT)

print("gestures:", GESTURES)
print("images:", sum(1 for _ in HAGRID_ROOT.rglob("*.jpg")))
print("annotation files:", sorted(p.name for p in ANNOTATIONS.rglob("*.json"))[:5], "...")
print(
    "hand-keypoints train images:",
    sum(1 for _ in (ULTRALYTICS_ROOT / "images" / "train").glob("*.jpg")),
)

In [ ]:
import json
from pathlib import Path

from cpsa_ml.datasets.hands import (
    HagridSource,
    HandBuildConfig,
    UltralyticsHandSource,
    build_hand_dataset,
)

OUT = Path("/kaggle/working/hand21")
USE_ULTRALYTICS = True  # False -> HaGRID-only dataset, no CC BY-NC-SA source
VAL_RATIO = 0.05  # share of the HaGRID train images held out for validation
SEED = 0

report = build_hand_dataset(
    HandBuildConfig(out=OUT, imgsz=192, val_ratio=VAL_RATIO, seed=SEED),
    hagrid=HagridSource(images=HAGRID_ROOT, annotations=ANNOTATIONS),
    ultralytics=(UltralyticsHandSource(root=ULTRALYTICS_ROOT) if USE_ULTRALYTICS else None),
)
print(json.dumps(report.to_dict(), indent=2))

In [ ]:
print((OUT / "data.yaml").read_text())
print(json.dumps(json.loads((OUT / "stats.json").read_text())["splits"], indent=2))

stats = json.loads((OUT / "stats.json").read_text())
print("licenses:", stats["licenses"])
assert stats["kpt_shape"] == [21, 3]
assert stats["splits"]["train"]["images_written"] > 0
if "license_warning" in stats:
    # The Ultralytics hand-keypoints source is CC BY-NC-SA 4.0: the built crops
    # inherit a non-commercial restriction. Set USE_ULTRALYTICS = False and
    # rebuild before publishing the dataset for anything commercial.
    print("LICENSE WARNING:", stats["license_warning"])

In [ ]:
from pathlib import Path

from cpsa_ml.models import config_path
from cpsa_ml.train import TrainConfig, train

VARIANT = "n"
ACTIVATION = "lrelu"  # DPU-friendly LeakyReLU(26/256)
EPOCHS = 100
BATCH = 64
DEVICE = "0,1"  # 2x T4 DDP
IMGSZ = 192
PROJECT = Path("/kaggle/working/runs")
NAME = "hand21-yolov8n-lrelu"
RESUME = ""  # e.g. "/kaggle/input/cpsa-hand-model/last.pt"

best = train(
    TrainConfig(
        data=OUT / "data.yaml",
        schema="hand21",
        variant=VARIANT,
        activation=ACTIVATION,
        model_cfg=config_path("yolov8n-hand-lrelu.yaml"),
        epochs=EPOCHS,
        batch=BATCH,
        device=DEVICE,
        imgsz=IMGSZ,
        project=PROJECT,
        name=NAME,
        resume=Path(RESUME) if RESUME else None,
        seed=0,
        workers=4,
        patience=30,
        amp=True,
        val=True,
        plots=True,
    )
)
print("best checkpoint:", best)

In [ ]:
import shutil
from pathlib import Path

MODEL = Path("/kaggle/working/hand-model")
MODEL.mkdir(parents=True, exist_ok=True)
shutil.copy2(best, MODEL / "best.pt")
for name in ("data.yaml", "schema.json", "stats.json"):
    shutil.copy2(OUT / name, MODEL / name)
(MODEL / "train_args.json").write_text(
    (PROJECT / NAME / "args.yaml").read_text(encoding="utf-8"), encoding="utf-8"
)

for path in sorted(MODEL.iterdir()):
    print(path.name, path.stat().st_size)

## Next steps

Re-publish `/kaggle/working/hand21` as the dataset `fabiocabeccia/cpsa-hand21`
(only if the CC BY-NC-SA source is acceptable) and `/kaggle/working/hand-model`
as `fabiocabeccia/cpsa-hand-model`.  The deployment export goes through
`cpsa_ml.onnx_check.export_onnx(best, ..., imgsz=192)` and the DPU audit at
192x192; the runtime feeds the crop produced by the stage-1 hand box.

Booth recordings can be added to the same dataset without Kaggle:

```
ml/.venv/bin/python -m cpsa_ml.pseudolabel --frames data/booth/P01_... --out data/labeled/P01
```

which writes full-frame `HALFBUST_62` labels and `hand21` crops (with the
handedness recorded) from the RTMW teacher.  Keep those frames local: they
contain faces.
